# Bag of Words, TF-IDF, and Text Representation Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: build the vocabulary

In [ ]:
```python

def build_vocab(docs):

    vocab = {}

    for doc in docs:

        for token in doc:

            if token not in vocab:

                vocab[token] = len(vocab)

    return vocab

In [ ]:
```

Input: list of tokenized documents (any word-level tokenizer will do; the `code/main.py` in this lesson uses a simplified lowercase variant). Output: `{word: index}` dict. Stable insertion order means word index 0 is the first word seen in the first document. Convention varies; scikit-learn sorts alphabetically.

### Step 2: bag of words

In [ ]:
```python

def bag_of_words(docs, vocab):

    matrix = [[0] * len(vocab) for _ in docs]

    for i, doc in enumerate(docs):

        for token in doc:

            if token in vocab:

                matrix[i][vocab[token]] += 1

    return matrix

In [ ]:
```

In [ ]:
```python

>>> docs = [["cat", "sat", "on", "mat"], ["cat", "cat", "ran"]]

>>> vocab = build_vocab(docs)

>>> bag_of_words(docs, vocab)

[[1, 1, 1, 1, 0], [2, 0, 0, 0, 1]]

In [ ]:
```

Rows are documents. Columns are vocabulary indices. Entry `[i][j]` is "how many times word `j` appears in document `i`." Doc 1 has `cat` twice because it did. Doc 0 has `ran` zero times because it did not.

### Step 3: term frequency and document frequency

In [ ]:
```python

import math

def term_frequency(doc_bow, doc_length):

    return [c / doc_length if doc_length else 0 for c in doc_bow]

def document_frequency(bow_matrix):

    df = [0] * len(bow_matrix[0])

    for row in bow_matrix:

        for j, count in enumerate(row):

            if count > 0:

                df[j] += 1

    return df

def inverse_document_frequency(df, n_docs):

    return [math.log((n_docs + 1) / (d + 1)) + 1 for d in df]

In [ ]:
```

Two smoothing tricks worth naming. The `(n+1)/(d+1)` avoids `log(x/0)`. The trailing `+1` ensures a word in every document still has IDF 1 (not 0), matching scikit-learn's default. Other implementations use raw `log(N/df)`. Both work; the smoothed version is friendlier.

### Step 4: TF-IDF

In [ ]:
```python

def tfidf(bow_matrix):

    n_docs = len(bow_matrix)

    df = document_frequency(bow_matrix)

    idf = inverse_document_frequency(df, n_docs)

    out = []

    for row in bow_matrix:

        length = sum(row)

        tf = term_frequency(row, length)

        out.append([tf_j * idf_j for tf_j, idf_j in zip(tf, idf)])

    return out

In [ ]:
```

In [ ]:
```python

>>> docs = [

...     ["the", "cat", "sat"],

...     ["the", "dog", "sat"],

...     ["the", "cat", "ran"],

... ]

>>> vocab = build_vocab(docs)

>>> bow = bag_of_words(docs, vocab)

>>> tfidf(bow)

In [ ]:
```

Three documents, five vocab words (`the`, `cat`, `sat`, `dog`, `ran`). `the` appears in all three, so its IDF is low. `dog` appears in one, so its IDF is high. The vectors are sparse (most entries are small) and the discriminative words pop.

### Step 5: L2-normalize rows

In [ ]:
```python

def l2_normalize(matrix):

    out = []

    for row in matrix:

        norm = math.sqrt(sum(x * x for x in row))

        out.append([x / norm if norm else 0 for x in row])

    return out

In [ ]:
```

Without normalization, a longer document gets a larger vector and dominates similarity scores. L2 normalization puts every document on the unit hypersphere. Cosine similarity between rows is now just a dot product.

## Exercises

In [ ]:
1. **Easy.** Implement `cosine_similarity(doc_vec_a, doc_vec_b)` on the L2-normalized TF-IDF output. Verify that identical documents score 1.0 and disjoint-vocabulary documents score 0.0.
2. **Medium.** Add `n-gram` support to `bag_of_words`. Parameter `n` produces counts over `n`-grams. Test that `n=2` on `["the", "cat", "sat"]` produces bigram counts for `["the cat", "cat sat"]`.
3. **Hard.** Build the TF-IDF-weighted-embedding hybrid above using GloVe 100d vectors (download once, cache). Compare classification accuracy against plain TF-IDF and plain mean-pooled embeddings on the 20 Newsgroups dataset. Report which wins where.